<a href="https://colab.research.google.com/github/varsityboi/speech-emotion-recognition/blob/main/notebooks/07_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07. Live demo with Gradio

This notebook loads the fine-tuned WavLM model and starts a small Gradio web app. You can record or upload a voice clip and see the predicted probability of each of the six emotions. The app can also match the loudness of the recording to the training data before predicting.

**Inputs** (Google Drive `MyDrive/SER_Project`):
- `models/wavlm_noscale_best.pt`: fine-tuned WavLM weights
- `logmel_index.csv`: used to pick example clips from the test speakers
- `waveforms_noscale.npy`: used for the example clips and the typical training loudness

**Outputs** (Google Drive `MyDrive/SER_Project`):
- Nothing is written to Drive. Six example clips are written to `/content/examples` in the Colab runtime.

### Setup and model loading
Installs Gradio, mounts Google Drive, rebuilds the WavLM classifier and loads the fine-tuned weights. Clips are handled at 16 kHz in windows of 56,000 samples (3.5 s), as in training.

In [1]:
!pip install -q gradio

from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd, torch, torch.nn as nn, librosa
from transformers import WavLMModel

PROJECT_DIR = "/content/drive/MyDrive/SER_Project"
EMOTIONS = ["angry", "disgust", "fear", "happy", "neutral", "sad"]
SR, N = 16000, 56000
device = "cuda" if torch.cuda.is_available() else "cpu"

class WavLMClassifier(nn.Module):
    def __init__(self, n_classes=6):
        super().__init__()
        self.wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base-plus", layerdrop=0.0)
        n_layers, hidden = self.wavlm.config.num_hidden_layers + 1, self.wavlm.config.hidden_size
        self.layer_weights = nn.Parameter(torch.zeros(n_layers))
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(hidden, 256), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(256, n_classes))
    def forward(self, wav):
        hs = torch.stack(self.wavlm(wav, output_hidden_states=True).hidden_states, dim=0)
        w = torch.softmax(self.layer_weights, dim=0)
        return self.head((w[:, None, None, None] * hs).sum(0).mean(1))

model = WavLMClassifier().to(device)
model.load_state_dict(torch.load(f"{PROJECT_DIR}/models/wavlm_noscale_best.pt", map_location=device))
model.eval()
print("WavLM loaded on", device)

Mounted at /content/drive


config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  378MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

WavLM loaded on cuda


### Build and launch the app
Computes the typical loudness (median RMS) of the training clips, then defines the same cleaning as in training with optional loudness matching. Clips longer than 3.5 s are cut into half-overlapping windows and the predictions are averaged. One test clip per emotion is added as an example, and the interface is launched with a public link.

In [3]:
import gradio as gr, soundfile as sf, os
import matplotlib.pyplot as plt, librosa.display

gr.close_all()

data = pd.read_csv(f"{PROJECT_DIR}/logmel_index.csv")
W = np.load(f"{PROJECT_DIR}/waveforms_noscale.npy", mmap_mode="r")

train_idx = np.where(data["split"] == "train")[0]
rms = [np.sqrt(np.mean(np.trim_zeros(np.asarray(W[i], dtype=np.float32), "b") ** 2)) for i in train_idx]
TARGET_RMS = float(np.median(rms))
print("Typical training loudness (RMS):", round(TARGET_RMS, 4))

def clean(path, match_loudness):
    y, _ = librosa.load(path, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)
    r = float(np.sqrt(np.mean(y ** 2)) + 1e-9)
    if match_loudness:
        y = y * (TARGET_RMS / r)
    return y, r

def windows(y):
    if len(y) <= N:
        return [np.pad(y, (0, N - len(y)))]
    starts = list(range(0, len(y) - N + 1, N // 2))
    if starts[-1] != len(y) - N:
        starts.append(len(y) - N)
    return [y[s:s + N] for s in starts]

def predict(audio_path, match_loudness):
    if audio_path is None:
        return None, None, "Record or upload a clip first."
    y, r = clean(audio_path, match_loudness)
    if len(y) < 0.5 * SR:
        return None, None, "Couldn't find enough speech. Try speaking a bit louder or longer."
    chunks = windows(y)
    x = torch.from_numpy(np.stack(chunks).astype(np.float32)).to(device)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.float16, enabled=(device == "cuda")):
        p = torch.softmax(model(x).float(), dim=1).mean(0).cpu().numpy()

    fig, axes = plt.subplots(2, 1, figsize=(9, 5))
    librosa.display.waveshow(y, sr=SR, ax=axes[0], color="steelblue")
    axes[0].set_title("Your voice (silence trimmed" + (", loudness matched)" if match_loudness else ")"))
    S = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_fft=400, hop_length=160, n_mels=128), ref=1.0)
    librosa.display.specshow(S, sr=SR, hop_length=160, x_axis="time", y_axis="mel", ax=axes[1], cmap="magma")
    axes[1].set_title("Spectrogram"); plt.tight_layout(); plt.close(fig)

    info = (f"Speech length: {len(y)/SR:.1f} s · {len(chunks)} window(s) of 3.5 s · "
            f"your loudness {r:.4f} vs training {TARGET_RMS:.4f}"
            + (" → matched" if match_loudness else " → NOT matched"))
    return {e: float(p[i]) for i, e in enumerate(EMOTIONS)}, fig, info

os.makedirs("/content/examples", exist_ok=True)
examples = []
for emo in EMOTIONS:
    i = data[(data["split"] == "test") & (data["emotion"] == emo)].sample(1, random_state=1).index[0]
    f = f"/content/examples/{emo}_{data.loc[i, 'file']}"
    sf.write(f, np.trim_zeros(np.asarray(W[i], dtype=np.float32), "b"), SR)
    examples.append([f, False])

demo = gr.Interface(
    fn=predict,
    inputs=[gr.Audio(sources=["microphone", "upload"], type="filepath", label="Record or upload your voice"),
            gr.Checkbox(value=True, label="Match loudness to training data (recommended for your own mic)")],
    outputs=[gr.Label(num_top_classes=6, label="Predicted emotion"),
             gr.Plot(label="What the model heard"),
             gr.Textbox(label="Info")],
    examples=examples,
    title="Speech Emotion Recognition",
    description=("Say a sentence with an emotion: angry, disgust, fear, happy, neutral or sad. "
                 "Model: WavLM fine-tuned on CREMA-D (73.9% on unseen speakers). "
                 "It listens to HOW you speak, not WHAT you say."),
    flagging_mode="never",
)
demo.launch(share=True, debug=True)

Closing server running on port: 7860
Typical training loudness (RMS): 0.0219
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://90aa8ef34c2bd34f4b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://90aa8ef34c2bd34f4b.gradio.live


The public gradio.live link in the output expires when the Colab session ends; re-run cells 1 and 2 to get a new one.